# HanziGen - GBK 补字推理版（训练完成后的最终路线）

> 本版聚焦**最终路线**，适合已经完成 VQ-VAE / LDM 训练、只需补字生成的情况：
>
> ```
> Cell 0: 配置字体名 + 补字基准
> Cell 1: 环境初始化 / 断连自检（可重复运行，已就绪自动跳过）
> Cell 2: 生成 GBK 缺失字清单 + 改写 inference.sh
> Cell 3: 推理生成 + 指标 + 转 SVG
> Cell 4: 下载 SVG zip → FontForge 导入原字体 → 导出完整字体
> ```

## 现在只需要执行哪些 Cell？

| 你的情况 | 需要运行的 Cell |
|---|---|
| **上次会话还在运行**（环境、仓库、checkpoints 都在） | **Cell 0 → Cell 2 → Cell 3 → Cell 4** |
| **工作空间重新启动过** | **Cell 0 → Cell 1 → Cell 2 → Cell 3 → Cell 4** |

> - Cell 1 是"自检型"的，已就绪的部分会自动跳过，任何时候都可以先跑一遍求个安心
> - Cell 2 是核心新增：以 **GBK 简体标准字符集（20,902 字）** 为基准，算出你的字体缺失的字并写入 `inference.sh`
> - Cell 3 / 4 就是原来的推理 + 下载 SVG（无需重训模型）

---
## Cell 0: 配置参数

> **只改这里！** 填你放在 `fonts/` 目录下的字体文件名（英文）。

In [ ]:
# ==================== 修改你的字体文件名 ====================
TARGET_FONT = "ChangguMingtiMedium.otf"   # 改成你放在 fonts/ 目录下的字体名
# ==========================================================

# ===== 补字基准选择（二选一）=====

# 注意：GB2312 的正确写法是 gb2312（不是 gbk2312）
# "gbk"    = GBK 简体标准字符集（20,902 字，简体+常用繁体，推荐）
# "gb2312" = 仅 GB2312 简体核心字（6,763 字，范围更小更保守）
CHARSET_BASE = "gbk"
# ==========================================================

FONT_NAME = TARGET_FONT.rsplit(".", 1)[0]
print(f"目标字体: {TARGET_FONT}")
print(f"字体名称: {FONT_NAME}")
print(f"补字基准: {CHARSET_BASE}")

---
## Cell 1: 环境初始化 + 断连自检（可重复运行）

> 检查/克隆仓库 → 安装依赖 → 检查字体 → 下载 Jigmo → 改写脚本 → 验证 GPU。
> **已就绪的部分会自动跳过**，断连重启后重跑本 Cell 即可恢复环境。

In [ ]:
import os, shutil, json, re, sys, glob, zipfile, io, urllib.request, subprocess
from fontTools.ttLib import TTFont

# ===== 0. 克隆项目代码（首次运行时自动拉取）=====
REPO_URL  = "https://github.com/ICW-k/HanziGen_ICWfork.git"
REPO_NAME = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")

is_project_root = os.path.isdir("scripts") and os.path.exists("requirements.txt")

if not is_project_root:
    found_repo = None
    for search_root in [os.getcwd(), "/workspace", "/home", "/"]:
        if not os.path.isdir(search_root):
            continue
        try:
            for entry in os.listdir(search_root):
                candidate = os.path.join(search_root, entry)
                if os.path.isdir(candidate) and os.path.isdir(os.path.join(candidate, "scripts")):
                    if os.path.exists(os.path.join(candidate, "requirements.txt")):
                        found_repo = candidate
                        break
        except PermissionError:
            continue
        if found_repo:
            break

    if found_repo:
        print(f"发现已有项目目录: {found_repo}")
        os.chdir(found_repo)
    else:
        print(f"正在克隆仓库: {REPO_URL}")
        subprocess.run(["git", "clone", REPO_URL], check=True)
        os.chdir(REPO_NAME)

    print(f"已切换到项目根目录: {os.getcwd()}")
else:
    print("已在项目根目录，跳过克隆")

# Cloud Studio 兼容：确保 python 命令指向 python3
print("\n===== 环境兼容检查 =====")
!which python3 && ln -sf $(which python3) /usr/local/bin/python 2>/dev/null; python --version && echo "python 命令已就绪"

# ===== 1. 确认工作目录 =====
PROJECT = os.getcwd()
print(f"\n工作目录: {PROJECT}")
!ls -F | head -30

# ===== 2. 安装依赖 =====
print("\n===== 安装 PyTorch + 依赖 =====")
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install -q -r requirements.txt
print("\n依赖安装完成！")

# ===== 3. 检查目标字体（智能定位）=====
print("\n===== 检查字体 =====")

def find_font_in_workspace(filename: str, search_root: str = ".") -> str | None:
    """在 search_root 下递归搜索 filename，返回第一个匹配的路径"""
    for root, dirs, files in os.walk(search_root, followlinks=False):
        dirs[:] = [d for d in dirs if not d.startswith(".") and d not in ("__pycache__", "node_modules")]
        for f in files:
            if f.lower() == filename.lower():
                return os.path.join(root, f)
    return None

font_path = f"fonts/{TARGET_FONT}"

print("当前 fonts/ 目录内容:")
if os.path.isdir("fonts"):
    !find fonts/ -type f 2>/dev/null | head -30
else:
    print("  [WARN] fonts/ 目录不存在！")
    parent_fonts = os.path.join(os.path.dirname(PROJECT) if os.path.dirname(PROJECT) != PROJECT else "..", "fonts")
    if os.path.isdir(parent_fonts):
        print(f"  但发现上级目录有: {parent_fonts}/")

if not os.path.exists(font_path):
    print(f"\n[WARN] {font_path} 不存在，正在搜索整个工作区...")
    found = find_font_in_workspace(TARGET_FONT, "/workspace" if os.path.isdir("/workspace") else ".")
    if found:
        print(f"[OK] 找到字体: {found}")
        os.makedirs("fonts", exist_ok=True)
        shutil.copy2(found, font_path)
        print(f"[OK] 已复制到: {font_path}")
    else:
        !ls -la
        raise FileNotFoundError(
            f"\n字体文件 '{TARGET_FONT}' 在整个工作区都找不到！\n"
            f"\n请检查:\n"
            f"  1. 文件名是否完全一致（注意大小写）? 当前配置: '{TARGET_FONT}'\n"
            f"  2. 字体是否已上传到 Cloud Studio 工作空间?\n"
            f"  3. 上传后文件是否放在 fonts/ 子目录下?"
        )
else:
    print(f"目标字体已就绪: {font_path}")

# ===== 4. 把字体路径写进所有 .sh 脚本 =====
print("\n===== 改写脚本字体路径 =====")
for sh_file in glob.glob("scripts/*.sh"):
    with open(sh_file, "r", encoding="utf-8") as f:
        content = f.read()
    content = re.sub(r'fonts/[\w.-]+\.(ttf|otf)', f'fonts/{TARGET_FONT}', content)
    with open(sh_file, "w", encoding="utf-8") as f:
        f.write(content)
print("脚本字体路径已统一替换")

# ===== 5. Jigmo 参考字体：从官方 ZIP 下载 =====
print("\n===== 准备 Jigmo 参考字体 =====")

jigmo_files = ["jigmo.ttf", "jigmo2.ttf", "jigmo3.ttf"]

def download_jigmo_fonts(target_dir="fonts/jigmo"):
    os.makedirs(target_dir, exist_ok=True)
    zip_url = "https://kamichikoichi.github.io/jigmo/Jigmo-20250912.zip"
    print(f"  下载 Jigmo ZIP: {zip_url}")
    try:
        resp = urllib.request.urlopen(zip_url, timeout=30)
        data = resp.read()
        if len(data) < 10000:
            raise ValueError(f"下载数据太小 ({len(data)} bytes)")
        zf = zipfile.ZipFile(io.BytesIO(data))
    except Exception as e:
        print(f"  [ERROR] ZIP 下载失败: {e}")
        return False
    for zip_name in zf.namelist():
        basename = os.path.basename(zip_name).lower()
        if basename in jigmo_files:
            zf.extract(zip_name, target_dir)
            extracted = os.path.join(target_dir, zip_name)
            target = os.path.join(target_dir, basename)
            if extracted != target:
                if os.path.exists(target):
                    os.remove(target)
                os.rename(extracted, target)
    zf.close()
    return True

def validate_font_file(fpath):
    try:
        f = TTFont(fpath)
        if "cmap" not in f:
            return False, "缺少 cmap 表"
        return True, f"OK ({len(f.getBestCmap())} glyphs)"
    except Exception as e:
        return False, str(e)[:80]

need_download = False
for fname in jigmo_files:
    fpath = f"fonts/jigmo/{fname}"
    if os.path.exists(fpath):
        valid, msg = validate_font_file(fpath)
        if not valid:
            print(f"  [WARN] {fname} 无效 ({msg})")
            os.remove(fpath)
            need_download = True
    else:
        need_download = True

if need_download:
    if not download_jigmo_fonts():
        raise RuntimeError("Jigmo 下载失败！")
    print("Jigmo 字体下载完成，验证中...")
    for fname in jigmo_files:
        valid, msg = validate_font_file(f"fonts/jigmo/{fname}")
        print(f"    [{'OK' if valid else 'ERROR'}] {fname}: {msg}")
        if not valid:
            raise RuntimeError(f"Jigmo 字体 {fname} 验证失败！")
else:
    print("Jigmo 字体已就绪，跳过下载")

# ===== 6. 断连自检 =====
print("\n===== 断连自检 =====")
os.makedirs("checkpoints", exist_ok=True)

vqvae_ckpt = f"checkpoints/vqvae_{FONT_NAME}.pth"
ldm_ckpt = f"checkpoints/ldm_{FONT_NAME}.pth"
print(f"  VQ-VAE 检查点:   {'存在: '+vqvae_ckpt if os.path.exists(vqvae_ckpt) else '不存在'}")
print(f"  LDM 检查点:      {'存在: '+ldm_ckpt if os.path.exists(ldm_ckpt) else '不存在'}")
if not os.path.exists(ldm_ckpt):
    print("\n  [WARN] LDM 检查点不存在！若你尚未完成训练，请回到完整版 notebook 完成 Cell 2-4。")

# ===== 7. 验证 GPU =====
print("\n===== 验证 GPU =====")
import torch
assert torch.cuda.is_available(), "GPU 不可用！请在 Cloud Studio 工作空间设置中切换到 GPU 实例"
prop = torch.cuda.get_device_properties(0)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"显存: {prop.total_memory / 1024**3:.1f} GB")
print(f"CUDA: {torch.version.cuda}")
print(f"PyTorch: {torch.__version__}")
print(f"\n全部初始化完成！字体: {font_path}")

---
## Cell 2: 生成补字字符集（GBK 缺失字）+ 改写 inference.sh

> 核心步骤：以 **GBK（20,902 字）** 为基准，算出你的字体缺失的字，写入字符集文件，并让 `inference.sh` 指向它。
>
> - 生成结果 = GBK 汉字 − 你的字体已覆盖的字（简体为主）
> - 字体已有的字**不会被重新生成**（保留原字形，质量无损）
> - 若想只补简体核心字，在 Cell 0 把 `CHARSET_BASE` 改为 `"gb2312"`

In [ ]:
import os, re
from fontTools.ttLib import TTFont

font_path = f"fonts/{TARGET_FONT}"

# ===== 1. 校验并构建基准字符集（GBK 或 GB2312，系统自带编码器，无需额外文件）=====
_CHARSET_ALIAS = {
    "gbk": "gbk",
    "gb2312": "gb2312",
    "gbk2312": "gb2312",   # 兼容手误（GB2312 的正确写法是 gb2312）
    "GBK": "gbk",
    "GB2312": "gb2312",
}
if CHARSET_BASE not in _CHARSET_ALIAS:
    raise ValueError(
        f"CHARSET_BASE 取值错误: {CHARSET_BASE!r}\n"
        f"可选值只有: \"gbk\"（推荐）或 \"gb2312\"\n"
        f"提示：GB2312 的正确写法是 gb2312，不是 gbk2312")
enc = _CHARSET_ALIAS[CHARSET_BASE]
CHARSET_BASE = enc   # 归一化，避免别名/大小写影响后续目录名
base_chars = set()
for cp in range(0x4E00, 0xA000):          # CJK 基本区全部码位
    try:
        chr(cp).encode(enc)                # 能按该编码集编码 = 属于该字符集
        base_chars.add(chr(cp))
    except UnicodeEncodeError:
        pass
print(f"[{CHARSET_BASE}] 基准字符集: {len(base_chars)} 字")

# ===== 2. 读字体已覆盖的码位 =====
font = TTFont(font_path, fontNumber=0)
cmap = set()
for table in font["cmap"].tables:
    if table.isUnicode():
        cmap.update(table.cmap.keys())
print(f"字体总码位: {len(cmap)}")

# ===== 3. 缺失 = 基准字符集 - 字体已有 =====
missing = sorted(base_chars - {chr(c) for c in cmap})
print(f"{CHARSET_BASE} 中字体缺失: {len(missing)} 字")
print("前 20 个缺失字:", "".join(missing[:20]))

# ===== 4. 写入自定义字符集 =====
out_dir = f"charsets/{CHARSET_BASE}_coverage/{FONT_NAME}"
os.makedirs(out_dir, exist_ok=True)
out_path = f"{out_dir}/missing.txt"
with open(out_path, "w", encoding="utf-8") as f:
    f.write("\n".join(missing))
print(f"[OK] 已写入 {out_path}")

# ===== 5. 改写 inference.sh：指向新字符集 + 切回 GPU =====
with open("scripts/inference.sh", encoding="utf-8") as f:
    content = f.read()
content = re.sub(r'CHARSET_PATH="[^"]*"', f'CHARSET_PATH="{out_path}"', content)
content = re.sub(r'DEVICE="[^"]*"', 'DEVICE="cuda"', content)
with open("scripts/inference.sh", "w", encoding="utf-8") as f:
    f.write(content)
print("[OK] inference.sh 已指向新字符集并切回 GPU")
print("\n下一步：运行 Cell 3 开始推理生成（13,000+ 字建议预留 30-60 分钟 GPU 机时）")

---
## Cell 3: 推理生成 + 指标 + 转换 SVG

> 依赖 LDM 训练完成（`checkpoints/ldm_{FONT_NAME}.pth` 存在）。
> 生成范围 = Cell 2 写入的 GBK 缺失字清单。

In [ ]:
import os, subprocess

ldm_ckpt = f"checkpoints/ldm_{FONT_NAME}.pth"

if not os.path.exists(ldm_ckpt):
    print(f"{ldm_ckpt} 不存在，请先完成模型训练（完整版 notebook Cell 2-4）")
else:
    print("\n===== 推理生成 =====")
    subprocess.run(["bash", "scripts/inference.sh"], check=True)
    print("\n===== 计算评估指标 =====")
    subprocess.run(["bash", "scripts/compute_metrics.sh"], check=True)
    print("\n===== 转换 SVG =====")
    subprocess.run(["bash", "scripts/convert_to_svg.sh"], check=True)
    print("\n===== 全部完成！=====")
    print(f"生成的 SVG 文件位置: svgs_{FONT_NAME}/")
    print(f"生成的样本位置: samples_{FONT_NAME}/")

---
## Cell 4: 导出并下载 SVG 结果

> 把 `svgs_{FONT_NAME}/` 打包成 zip → 生成浏览器下载链接 + 同步保存到工作空间根目录（文件树可见处）+ 页面内预览前 9 个。
>
> 下载后即可用 **FontForge**：打开原字体 → Import 这些 SVG → Generate 导出完整字体。

In [ ]:
import os, glob, zipfile, io, base64
from IPython.display import HTML, display

svg_dir = f"svgs_{FONT_NAME}"
if not os.path.isdir(svg_dir):
    print(f"{svg_dir} 不存在，请先运行 Cell 3 完成推理与转换")
else:
    svg_files = sorted(glob.glob(os.path.join(svg_dir, "*.svg")))
    print(f"共找到 {len(svg_files)} 个 SVG 文件")

    # 1. 打包成 zip（内存中生成）
    zip_buffer = io.BytesIO()
    with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in svg_files:
            zf.write(f, arcname=os.path.basename(f))
    zip_data = zip_buffer.getvalue()
    zip_name = f"svgs_{FONT_NAME}.zip"

    # 2. 同步保存到工作空间根目录（文件树可见）
    saved_paths = []
    roots = [os.path.abspath(os.getcwd()), os.path.dirname(os.path.abspath(os.getcwd())),
             "/workspace", "/home", "/root"]
    seen = set()
    for r in roots:
        if r and r not in seen and os.path.isdir(r):
            seen.add(r)
            try:
                p = os.path.join(r, zip_name)
                with open(p, "wb") as f:
                    f.write(zip_data)
                saved_paths.append(p)
            except Exception:
                pass
    for p in saved_paths:
        print(f"  [已保存] {p} ({len(zip_data)/1024:.0f} KB)")

    # 3. 浏览器内点击下载（base64 直链）
    b64 = base64.b64encode(zip_data).decode()
    download_link = (
        '<a href="data:application/zip;base64,' + b64 + f'" download="{zip_name}" '
        'style="display:inline-block;font-size:18px;font-weight:bold;color:#fff;'
        'background:#1a73e8;padding:12px 28px;border-radius:8px;'
        'text-decoration:none;">'
        f'下载全部 SVG ({len(svg_files)} 个 / {len(zip_data)/1024:.0f} KB)</a>'
    )
    display(HTML(download_link))

    # 4. 页面内预览前 9 个 SVG
    cards = []
    for f in svg_files[:9]:
        with open(f, encoding="utf-8") as fh:
            svg = fh.read()
        svg = svg.replace("<svg ", '<svg width="80" height="80" style="background:#fff;" ', 1)
        name = os.path.splitext(os.path.basename(f))[0]
        cards.append(
            f'<div style="border:1px solid #e0e0e0;border-radius:8px;padding:8px;'
            f'text-align:center;width:96px;">{svg}<div style="font-size:12px;color:#555;">{name}</div></div>'
        )
    display(HTML('<div style="display:flex;flex-wrap:wrap;gap:10px;">' + "".join(cards) + "</div>"))

---
## 断连恢复指南

工作空间重新启动后，按顺序运行：**Cell 0 → Cell 1 → Cell 2 → Cell 3 → Cell 4**。

- Cell 1 会自检环境、自动补齐缺失的依赖 / Jigmo / 字体路径
- Cell 2 会重新生成字符集清单（幂等，可重复运行）
- Cell 3 生成的 PNG 会**覆盖同名文件**，中断后重跑即可，不会混入旧数据

---

## 常见问题

| 问题 | 解决 |
|------|------|
| GPU 不可用 | Cloud Studio 工作空间设置中切换到 GPU 运行时 |
| LDM 检查点不存在 | 说明还没训练完，需回到完整版 notebook 完成训练 |
| 生成太慢 | 可把 `scripts/inference.sh` 的 `SAMPLE_STEPS=50` 改为 `20`（速度翻倍，质量略降） |
| 想只补简体核心字 | Cell 0 把 `CHARSET_BASE` 改为 `"gb2312"` 再重跑 Cell 2 |
| 想扩大范围到生僻字 | 把 Cell 2 的基准换成 unihan basic + ext_a（见对话方案） |